In [2]:
import os
%load_ext autoreload
%autoreload 1
%aimport utility.functions
%aimport src.analysis
%aimport src.dataset
%aimport src.plots
%aimport src.simulation
import numpy as np
import utility.functions as fn
import src.simulation as sim
import src.analysis as an
import src.dataset as ds
import src.plots as plots
import torch
from scipy.stats import norm
import pandas as pd
from utility.double_beta_spectrum import pdf_ratio2b
import matplotlib.pyplot as plt
channels = [3, 5, 9, 11, 13, 15, 17, 19]
channel_ID = [5, 9, 11, 10, 4, 2, 12, 3]

Main loop to compute the BI for all channels and acceptance levels

In [ ]:
# Initialize output DataFrame
out_data = pd.DataFrame(columns=["channel_ID", "channel", "acceptance", "BI", "rp","J"])
out_data.to_csv('../outputs/BI_results_interp.csv', index=False)
n_deriv = 0

for index_channel in np.argsort(channel_ID):
    channel = channels[index_channel]
    
    # Load datasets and channel specifications
    data_single, data_pileup, _, _, _ = ds.create_data_sets(channel, win_length=2000, shift=0, pos_prefix="_bis")
    #data_single, data_pileup = ds.create_data_sets_denoise(channel, win_length=2000, shift=0, pos_prefix="_bis")
    H_unit, S, nps, w, pulse_factor = ds.get_channel_specs(channel, n_deriv=n_deriv, window_fct = np.hanning) #, nps_name="denoised_noise"
    
    signal_amp = ds.get_amp_Q_val(channel)*pulse_factor
    t_min, t_max, N_t = 0, 8e-4, 100
    r_min, r_max, N_r = 0., .5, 25
    
    ratio_distribution = pdf_ratio2b(np.linspace(r_min, r_max, N_r))
    ratio_distribution /= np.mean(ratio_distribution)
    
    # Convert data to PyTorch tensors
    S_torch = torch.tensor(S, dtype=torch.cfloat)
    H_unit_torch = torch.tensor(H_unit, dtype=torch.cfloat)
    w_torch = torch.tensor(w, dtype=torch.cfloat)
    t_torch = torch.linspace(t_min, t_max,N_t, dtype=torch.cfloat)
    r_torch = torch.linspace(r_min, r_max, N_r, dtype=torch.cfloat)
    nps_torch = torch.tensor(nps, dtype=torch.cfloat)
    signal_amp_torch = torch.tensor(signal_amp, dtype=torch.float32)
    ratio_distribution_torch = torch.tensor(ratio_distribution, dtype=torch.cfloat)

    # Loop over acceptance levels
    # acceptances = [0.7, 0.8, 0.84, 0.87, 0.9, 0.92, 0.93, 0.94, 0.95, 0.96, 0.99]
    acceptances = [0.9]
    for acceptance in acceptances:
        N_sigma = norm.ppf(1 - (1 - acceptance) * 100 / 100)
        # f1_0, f2_0 = np.load(f"../outputs/Pileup_filter_functions/channel_{channel}/functions_eff{int(acceptance*100):d}_d0.npy")
        # print(torch.tensor(f1_0,dtype=torch.float).size())
        f1_opt, f2_opt, J_values = an.optimize_filters(S_torch, H_unit_torch, w_torch, t_torch, r_torch, nps_torch, signal_amp_torch,ratio_distribution_torch, N_sigma=N_sigma, n_trials=500, verbose=True)#, f1_init=torch.tensor(f1_0,dtype=torch.float), f2_init=torch.tensor(f2_0,dtype=torch.float))
        #save_filter_function 
        np.save(f"../outputs/training_Js/channel_{channel}/J_{int(acceptance*100):d}_d0", np.array(J_values))
        np.save(f"../outputs/Pileup_filter_functions/channel_{channel}/functions_eff{int(acceptance*100):d}_d0", np.array([f1_opt, f2_opt]))
        # Compute PSD and BI
        BI, rp, PSD_pileup, PSD_single, Amp_1_pileup, Amp_2_pileup, Amp_1_single, Amp_2_single = an.compute_BI(data_pileup, data_single, acceptance, H_unit, f1_opt, f2_opt,window_fct = np.hanning,n_deriv = n_deriv,full_output = True)
        sigma_rp, sigma_BI = an.compute_BI_uncertainty(data_pileup, data_single, acceptance, H_unit, f1_opt, f2_opt, n_deriv=n_deriv, n_toys=100)
        # Append results to DataFrame
        out_data = pd.concat([out_data, pd.DataFrame([{"channel_ID": channel_ID[index_channel], "channel": channel, 
                                                       "acceptance": acceptance, "BI": BI, "rp": rp, "J": J_values[-1]}])], ignore_index=True)
        # Save results to CSV
        out_data.to_csv('../outputs/BI_results_interp_d0.csv', index=False)
        print(f"Channel {channel}, Acceptance {acceptance}: J={J_values[-1]}, 1-rp={1-rp}, BI={BI}")


In [19]:
channel = 9
n_deriv = 0
for channel in channels:
    data_single, data_pileup, _, _, _ = ds.create_data_sets(channel, win_length=2000, shift=0, pos_prefix="_bis")
    H_unit, S, nps, w, pulse_factor = ds.get_channel_specs(channel, n_deriv=n_deriv, window_fct = np.hanning) #, nps_name="denoised_noise"
    acceptance = 0.9
    f1_opt, f2_opt = np.load(f"../outputs/Pileup_filter_functions/channel_{channel}/functions_eff{int(acceptance*100):d}_d0.npy")
    BI, rp, PSD_pileup, PSD_single, Amp_1_pileup, Amp_2_pileup, Amp_1_single, Amp_2_single = an.compute_BI(data_pileup, data_single, acceptance, H_unit, f1_opt, f2_opt,window_fct = np.hanning,n_deriv = n_deriv,full_output = True)
    sigma_rp, sigma_BI = an.compute_BI_uncertainty(PSD_single, PSD_pileup, acceptance, rp)
    print(f"Channel {channel}, Acceptance {acceptance}: 1-rp={1-rp:.3f} +/- {sigma_rp:.3f} , BI={BI*1e5:.2f} +/- {sigma_BI*1e5:.2f} 1e-5 ckky")

Channel 3, Acceptance 0.9: 1-rp=0.182 +/- 0.003 , BI=4.80 +/- 0.09 1e-5 ckky
Channel 5, Acceptance 0.9: 1-rp=0.210 +/- 0.003 , BI=5.53 +/- 0.09 1e-5 ckky
Channel 9, Acceptance 0.9: 1-rp=0.220 +/- 0.004 , BI=5.80 +/- 0.09 1e-5 ckky
Channel 11, Acceptance 0.9: 1-rp=0.193 +/- 0.003 , BI=5.10 +/- 0.09 1e-5 ckky
Channel 13, Acceptance 0.9: 1-rp=0.179 +/- 0.003 , BI=4.71 +/- 0.09 1e-5 ckky
Channel 15, Acceptance 0.9: 1-rp=0.278 +/- 0.004 , BI=7.33 +/- 0.11 1e-5 ckky
Channel 17, Acceptance 0.9: 1-rp=0.212 +/- 0.004 , BI=5.59 +/- 0.09 1e-5 ckky
Channel 19, Acceptance 0.9: 1-rp=0.276 +/- 0.004 , BI=7.28 +/- 0.11 1e-5 ckky


Comparaison between analytical and simulated PSD for channel 9, acceptance 90%

In [ ]:
from src.plots import plot_biais_vs_dt
import matplotlib.pyplot as plt
S_H_delayed, S_H, S2_over_nps = an.precompute_constants(S_torch, H_unit_torch, w_torch, t_torch, nps_torch)
f1_opt, f2_opt = np.load(f"../outputs/Pileup_filter_functions/channel_{channel}/functions_eff90_V3.npy")
f1,f2  = torch.tensor(f1_opt,dtype=torch.cfloat), torch.tensor(f2_opt,dtype=torch.cfloat)
muY, sigmaY, mu1, mu2, var1, var2, cov12= an.compute_mu_sigma(f1, f2, S_H_delayed,r_torch, S_H, S2_over_nps, signal_amp, return_all=True)
mu = torch.mean(ratio_distribution_torch[:,None]*(1-muY.real), dim = 0).numpy().real
sigma = torch.sqrt(torch.mean(ratio_distribution_torch[:,None]*sigmaY**2, dim = 0)+torch.mean(ratio_distribution_torch[:,None]*(1-muY.real)**2, dim = 0)-mu**2).numpy().real
fig,ax,ax1=plot_biais_vs_dt(PSD_single.real, PSD_pileup.real, dt_pileup,cut=1-cut,bins=20)
ax.fill_between(t_torch.numpy().real*1e3, (mu-sigma).real, (mu+sigma).real, alpha=0.2, color='blue', label='Analytical PSD $\pm$ 1 std')
ax.plot(t_torch.numpy().real*1e3, mu.real, color='blue', label='Analytical PSD mean')
ax.set_xlim(0,0.8)
ax.set_ylim(-0.003,0.015)
ax.legend()
plt.show()

This is a test to study the impact of the Bessel filter cutoff frequency on the filter optimization and on the final BI.

In [ ]:
from src.simulation import bessel_filter_coeffs, filtered_pulse_model, pulse_model
from lmfit import Model
from scipy.signal import freqz

freq = np.fft.rfftfreq(2000, d=1e-4)  # Hz
omega = 2 * np.pi * freq  # rad/s
fs = 1e4
t = np.arange(0,2000)/fs
n_deriv = 0
acceptance = 0.9
N_sigma = norm.ppf(1 - (1 - acceptance) * 100 / 100)

model = Model(filtered_pulse_model,independent_vars=['t'],param_names=['A','t0','tau_rise','tau_decay1','tau_decay2','p'])
params = model.make_params(A=1,t0=0.1, tau_rise=1e-4,tau_decay1=1e-3,tau_decay2=2e-3,p=0.5)

_, h_bessel_small_2500 = freqz(*bessel_filter_coeffs(6, 2500, 1e4), worN=1001)
h_bessel_2500 = np.concatenate([h_bessel_small_2500, np.conj(h_bessel_small_2500[1:1000][::-1])])
for channel in [9]:    
    H_unit, S, nps, w, pulse_factor = ds.get_channel_specs(channel, n_deriv=0, window_fct = np.hanning)
    
    w_torch = torch.tensor(w, dtype=torch.cfloat)
    t_torch = torch.linspace(0, 8e-4, 100, dtype=torch.cfloat)
    r_torch = torch.linspace(0., .5, 25, dtype=torch.cfloat)
    signal_amp = ds.get_amp_Q_val(channel)*pulse_factor
    signal_amp_torch = torch.tensor(signal_amp, dtype=torch.float32)
    ratio_distribution = pdf_ratio2b(r_torch.numpy())
    ratio_distribution /= np.mean(ratio_distribution)
    ratio_distribution_torch = torch.tensor(ratio_distribution, dtype=torch.cfloat)
    nps_torch = torch.tensor(nps, dtype=torch.cfloat)
    
    S_torch = torch.tensor(S, dtype=torch.cfloat)
    H_unit_torch = torch.tensor(H_unit, dtype=torch.cfloat)
    f1_opt, f2_opt, J_values = an.optimize_filters(S_torch, H_unit_torch, w_torch, t_torch, r_torch, nps_torch, signal_amp_torch,ratio_distribution_torch, N_sigma=N_sigma, n_trials=500, verbose=False)
    
    result = model.fit(np.fft.ifft(S),params,t=t, order=6,fc=2500,nan_policy='omit')  
    S_bis = np.fft.fft(filtered_pulse_model(t, **result.params,order=6,fc=2500))
    H_bis = np.conj(S_bis) / nps
    H_unit_bis = H_bis / np.mean(np.abs(H_bis * S_bis))
    
    S_torch = torch.tensor(S_bis, dtype=torch.cfloat)
    nps_torch = torch.tensor(nps, dtype=torch.cfloat)
    H_unit_torch = torch.tensor(H_unit_bis, dtype=torch.cfloat)
    
    f1_opt_bis, f2_opt_bis, J_values_bis = an.optimize_filters(S_torch, H_unit_torch, w_torch, t_torch, r_torch, nps_torch, signal_amp_torch,ratio_distribution_torch, N_sigma=N_sigma, n_trials=500, verbose=False)
    

In [ ]:
fig, ax1 = plt.subplots(figsize=(8,6))
ax1.plot(freq, f1_opt[:1001],c="r",alpha=0.8,label='m1 & m2 with injected pulse')
ax1.plot(freq, f2_opt[:1001],c="r",alpha=0.8)
ax1.plot(freq, f1_opt_bis[:1001],c="b",alpha=0.8,label='m1 & m2 with Fitted pulse')
ax1.plot(freq, f2_opt_bis[:1001],c="b",alpha=0.8)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_ylabel('Filter Amplitude',fontdict = {'size':14})
ax1.set_xlabel('Frequency (Hz)',fontdict = {'size':14})
ax1.tick_params(axis='both', which='major', labelsize=12)
ax2 = ax1.twinx()
ax2.plot(freq, np.abs(S[:1001]),c="coral",label='Injected Pulse')
ax2.plot(freq, np.abs(S_bis[:1001]),c="dodgerblue",ls='--',label='Fitted Pulse')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_ylim(1e-4,100)
ax2.set_ylabel('Pulse Amplitude Spectrum', color='k',fontdict = {'size':14})
ax2.tick_params(axis='both', which='major', labelsize=12)

ax1.vlines(freq[np.argwhere(np.abs(h_bessel_2500)<1e-3)[0][0]],0,100,label='1e-3 Bessel reduction',colors='k',linestyles='--')
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, fontsize=12,loc='lower left')
# ax1.vlines(freq[np.argwhere(np.abs(h_bessel_2500)<1e-3)[0][0]],0,100,label='Cutoff Frequency',colors='k',linestyles='--')
plt.show()

In [ ]:

freq = np.fft.rfftfreq(2000, d=1e-4)  # Hz
omega = 2 * np.pi * freq  # rad/s
fs = 1e4
t = np.arange(0,2000)/fs
n_deriv = 0
acceptance = 0.9
N_sigma = norm.ppf(1 - (1 - acceptance) * 100 / 100)

model = Model(filtered_pulse_model,independent_vars=['t'],param_names=['A','t0','tau_rise','tau_decay1','tau_decay2','p'])
params = model.make_params(A=1,t0=0.1, tau_rise=1e-4,tau_decay1=1e-3,tau_decay2=2e-3,p=0.5)

_, h_bessel_small_2500 = freqz(*bessel_filter_coeffs(6, 2500, 1e4), worN=1001)
h_bessel_2500 = np.concatenate([h_bessel_small_2500, np.conj(h_bessel_small_2500[1:1000][::-1])])
fig,(ax1,ax2) = plt.subplots(2,1, figsize=(10,8), sharex=True, gridspec_kw={'height_ratios': [2, 1],'hspace':0})
for channel in [9]:    
    H_unit, S, nps, w, pulse_factor = ds.get_channel_specs(channel, n_deriv=0, window_fct = np.hanning)
    
    result = model.fit(np.fft.ifft(S),params,t=t, order=6,fc=2500,nan_policy='omit')
    ax1.plot(freq, np.abs(S)[:1001], label='Injected Pulse')
    S_fit = np.fft.rfft(filtered_pulse_model(t,**result.params,order=6,fc=2500))
    ax1.plot(freq, np.abs(S_fit), label='Fitted Pulse in time domain')
    ax1_bis = ax1.twinx()
    ax1_bis.plot(freq, np.abs(nps[:1001]), label='NPS',c="r",alpha=0.5)
    ax1_bis.set_yscale('log')
    ax1_bis.set_ylabel('Noise Power Spectrum', color='r',fontdict = {'size':14})
    ax1_bis.tick_params(axis='y', labelcolor='r', colors='r')
    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_ylim(1e-4,100)
    ax1.set_ylabel('Pulse Amplitude Spectrum',fontdict = {'size':14})
    ax1.legend(fontsize=12,loc='lower left')
    ax2.plot(freq, np.abs(S_fit-S[:1001]), label='Fit Residuals',ls="",marker='o',markersize=1)
    ax2.set_xscale('log')
    ax2.set_yscale('log')
    # ax2.set_ylim(0.,0.5)
    ax2.set_xlabel('Frequency (Hz)',fontdict = {'size':14})
    ax2.set_ylabel('Residual Amplitude',fontdict = {'size':14})
    ax2.legend(fontsize=12,loc='lower left')

In [ ]:
import struct
path = "/media/mp274748/Transcend/data/Aquisition_card/000037_20251202T164114_001_000.bin"
data = ds.BinaryDataset(path,win_length = 2000)
nps = an.create_NPS(data)
# pulse = filtered_pulse_model(t=np.arange(2000)/data.samplingRate, **params, order=6,fc=2500,nan_policy='omit')
# S_2500 = np.fft.fft(pulse*np.hanning(2000))
plt.plot(np.fft.rfftfreq(data.win_length,1/data.samplingRate),nps[:data.win_length//2+1])
plt.xscale('log')
plt.yscale('log')